# TODO 9.3.

- random sampling effect on RMSE
- induce missingness in sequence
- MAR assumption: generating coverage from the mag. field
- run KNN 

In [1]:
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

from pathlib import Path
from itertools import product
from sklearn.impute import KNNImputer
from numpy.lib.stride_tricks import sliding_window_view
from IPython.display import clear_output
from matplotlib.colors import LogNorm, TwoSlopeNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.dates import HourLocator, DateFormatter

from aipad.padimputer import PADImputer
from aipad.spacecrafts import SoloConstants, WindConstants
from aipad.datahandler import WindDataHandler
from aipad.pad_imputation import (
    make_train_test, calculate_scores, save_results, pad_histogram, induce_missingness)

In [2]:
bins = 8
time_avg_min = 1
avg_bin = f"{time_avg_min}min_{bins}bins"

file_path = Path("./data/hdf5") / f"wind_{avg_bin}.h5"
cov_path = Path("./data/coverages") / avg_bin
plot_path = Path(f"./plots/")

plot_path.mkdir(exist_ok=True)

solo = SoloConstants()
wind = WindConstants()

In [3]:
def convert_timestamps(attrs):
    """Convert Epoch timestamps to NumPy datetime."""
    epoch = attrs["Epoch"]
    t_unit = attrs["t_unit"]
    return epoch.astype(f"datetime64[{t_unit}]")

def load_event_data(hdf, event_name, sc):
    """Load datasets and attributes for a given event."""
    event_group = hdf[event_name]

    # Load datasets
    intensity_attrs = dict(event_group["Intensity"].attrs)
    intensity_attrs["Epoch"] = convert_timestamps(intensity_attrs)
    intensity = pd.DataFrame(event_group["Intensity"][:], index=intensity_attrs["Epoch"], columns=sc.sectors)

    mag_field_attrs = dict(event_group["MagField"].attrs)
    mag_field_attrs["Epoch"] = convert_timestamps(mag_field_attrs)
    mag_field = pd.DataFrame(event_group["MagField"][:], index=mag_field_attrs["Epoch"], columns=["Bx", "By", "Bz", "B"])

    ind = pd.MultiIndex.from_product([sc.sectors, ["min", "center", "max"]])
    coverage_attrs = dict(event_group["Coverage"].attrs)
    coverage_attrs["Epoch"] = convert_timestamps(coverage_attrs)
    coverage = pd.DataFrame(event_group["Coverage"][:], index=coverage_attrs["Epoch"], columns=ind)

    return {
        "Intensity": intensity,
        "MagField": mag_field,
        "Coverage": coverage,
        "Intensity_attrs": intensity_attrs,
        "MagField_attrs": mag_field_attrs,
        "Coverage_attrs": coverage_attrs
    }

def target_from_prediction(true, reduced, imputed):
    pred = np.where((np.isnan(reduced)
                    & np.isfinite(true)
                    & np.meshgrid(np.isfinite(reduced).any(axis=1), np.arange(0, reduced.shape[1]),
                                  indexing="ij")[0]),
                    imputed, np.nan)
    target = np.where(np.isfinite(pred), true, np.nan)
    return target, pred

def plot_results(pa, I_data, B_data, true, reduced, imputed, score: str,
                 sc: WindConstants, cov_sc: SoloConstants,
                 save_plot=True, save_path=None) -> None:
    
    target, pred = target_from_prediction(true, reduced, imputed)

    # Score over whole 12 hrs
    total_score = calculate_scores(target, pred, score)

    true_miss_percent = np.sum(np.where(np.isnan(true), 1, 0)) \
        / (true.shape[0] * true.shape[1]) * 100
    reduced_miss_percent = np.sum(np.where(np.isnan(reduced), 1, 0)) \
        / (reduced.shape[0] * reduced.shape[1]) * 100

    X, Y = np.meshgrid(I_data.index.values, pa, indexing="ij")

    diff = target - pred

    norm = LogNorm(np.nanmin(true), np.nanmax(true))

    fig, axs = plt.subplots(nrows=8, figsize=(16, 32), sharex=True)

    for b in range(B_data.values.shape[1]):
        axs[0].plot(B_data.index.values, B_data.iloc[:, b], label=B_data.columns[b])

    for i in range(I_data.shape[1]):
        axs[1].plot(I_data.index.values, I_data.iloc[:, i], label=sc.sectors[i])

    axs[0].legend(loc="upper right")
    axs[0].set_ylabel("B [nT]")
    axs[0].xaxis.set_major_locator(HourLocator(range(24)))
    axs[0].xaxis.set_major_formatter(DateFormatter("%d %b\n%H:%M"))
    axs[0].set_title("Magnetic field (GSE coordinates)")
    
    axs[1].set_yscale("log")
    axs[1].legend(loc="upper right")
    axs[1].set_ylabel(r"I $[\mathrm{(cm^2\ s\ sr\ MeV)^{-1}}]$")
    axs[1].set_title("Intensities")

    axs[2].pcolormesh(X, Y, true, norm=norm, cmap="inferno")
    axs[2].set_title(f"{sc.name.upper()} PAD, total {true_miss_percent:.2f} % missing")
    axs[2].set_yticks(np.linspace(0, 180, 9))
    axs[2].set_ylabel(f"Pitch angle [{u"\u03b8"}]")

    axs[3].pcolormesh(X, Y, reduced, norm=norm, cmap="inferno")
    axs[3].set_title(f"{sc.name.upper()} PAD w/ reduction, {reduced_miss_percent:.2f} % missing")
    axs[3].set_yticks(np.linspace(0, 180, 9))
    axs[3].set_ylabel(f"Pitch angle [{u"\u03b8"}]")

    axs[4].pcolormesh(X, Y, imputed, norm=norm, cmap="inferno")
    axs[4].set_title("Imputed values")
    axs[4].set_yticks(np.linspace(0, 180, 9))
    axs[4].set_ylabel(f"Pitch angle [{u"\u03b8"}]")

    axs[5].pcolormesh(X, Y, pred, norm=norm, cmap="inferno")
    axs[5].set_title("Predicted values")
    axs[5].set_yticks(np.linspace(0, 180, 9))
    axs[5].set_ylabel(f"Pitch angle [{u"\u03b8"}]")

    axs[6].pcolormesh(X, Y, target, norm=norm, cmap="inferno")
    axs[6].set_title(f"Target values (RMSE = {total_score})")
    axs[6].set_yticks(np.linspace(0, 180, 9))
    axs[6].set_ylabel(f"Pitch angle [{u"\u03b8"}]")

    log_data = np.sign(diff) * np.log10(np.abs(diff) + 1e-10)  # Add small offset to avoid log(0)
    vmax = np.nanmax(log_data)
    vmin = np.nanmin(log_data)
    mesh = axs[7].pcolormesh(X, Y, log_data, norm=TwoSlopeNorm(vcenter=0, vmin=vmin, vmax=vmax), cmap="bwr")
    axs[7].set_title(r"log|target - prediction| (with sign of difference)")
    axins = inset_axes(axs[7], width="100%", height="100%", loc="center",
                       bbox_to_anchor=(1.01, 0, 0.03, 1), bbox_transform=axs[7].transAxes,
                       borderpad=0.2)
    fig.colorbar(mesh, cax=axins, ax=axs[7])
    axs[7].set_yticks(np.linspace(0, 180, 9))
    axs[7].set_ylabel(f"Pitch angle [{u"\u03b8"}]")
    axs[7].set_xlabel(f"Date (in {I_data.index[0].strftime("%Y")})")

    axs[0].set_xlim((I_data.index[0], I_data.index[-1]))
    if save_plot:
        plt.savefig(save_path)
        plt.close()

    else:
        plt.show()

### Simple imputation strategies: row/column mean, average of forward and backward fill, linear interpolation, KNN without train-test split


In [ ]:
miss_percents = [10, 20, 50, 80] 
repeats = 100
imputer = PADImputer()
index = pd.MultiIndex.from_product([list(imputer._strategy_map.keys()), ["time", "pitch_angle"]], 
                                    names=["method", "axis"])
columns = pd.MultiIndex.from_product([["rmse_mean", "rmse_std"], miss_percents],
                                    names=["stat", "miss_percent"])

# Axis = 0: mean/fill/interpolation column-wise (in the time direction), KNN samples are PA distributions
# Axis = 1: mean/fill/interpolation row-wise (in the angular direction), KNN samples are intensity time series
with h5py.File(file_path, 'r') as hdf:
    event_list = list(hdf.keys())

    for event_no in range(len(event_list)):
        if event_list[event_no].split("-")[0] < "20120407":
            continue
        print(f"Event {event_list[event_no]}")
        
        df = pd.DataFrame(index=index, columns=columns)
        event_data = load_event_data(hdf, event_list[event_no], wind)

        I_data = event_data["Intensity"]
        B_data = event_data["MagField"]
        cov_data = event_data["Coverage"]
        times = event_data["Intensity_attrs"]["Epoch"]
    
        X, Y, true = pad_histogram(wind, I_data.values, cov_data, 8)
        for miss_percent in miss_percents:
            for method, axis in product(imputer._strategy_map.keys(), ["time", "pitch_angle"]):
                #print(f"Running simple imputation: method = {method}, axis = {axis}")
                imputer.method = method
                imputer.axis = 0 if axis == "time" else 1
                scores = []
                for i in range(repeats):
                    reduced_cov, mask = induce_missingness(cov_data, miss_percent / 100)
                    X, Y, reduced = pad_histogram(wind, I_data.values, reduced_cov, 8)
                    imputed = imputer.fit_transform(reduced)
                    target, pred = target_from_prediction(true, reduced, imputed)
                    score = calculate_scores(target, pred, "rmse")
                    scores.append(score)
                    #print(f"RMSE: {calculate_scores(target, pred, "rmse"):.3f}")

                scores = np.array(scores)
                mean = scores.mean()
                std = scores.std(ddof=1)
                #print(f"Mean: {mean}, std: {std}\n")
                df.loc[(method, axis), ("rmse_mean", miss_percent)] = mean
                df.loc[(method, axis), ("rmse_std", miss_percent)] = std
                plot_dir = Path(f"./plots/imputers/{imputer.method}/{imputer.axis}")
                plot_dir.mkdir(exist_ok=True)
                plot_results(Y[0], I_data, B_data, true, reduced, imputed, "rmse", wind, solo,
                            save_path=plot_dir / f"{event_list[event_no]}_axis{imputer.axis}_{miss_percent}percent")

            df_sorted = df.sort_values(by=("rmse_mean", miss_percent))
            print(f"Results with {miss_percent}% MCAR missing values")
            print(f"\tBest: {df_sorted.index[0]} ({df_sorted["rmse_mean", miss_percent].iloc[0]:.3f} +- {df_sorted["rmse_std", miss_percent].iloc[0]:.3f})")
            print(f"\tWorst: {df_sorted.index[-1]} ({df_sorted["rmse_mean", miss_percent].iloc[-1]:.3f} +- {df_sorted["rmse_std", miss_percent].iloc[-1]:.3f})")
        csv_path = Path(f"./results")
        csv_path.mkdir(exist_ok=True)
        df.to_csv(csv_path / f"{event_list[event_no]}.csv")

        

Event 20120102-153731
Results with 10% MCAR missing values
	Best: ('interp', 'time') (12.866 +- 0.947)
	Worst: ('mean', 'time') (90.100 +- 8.353)
Results with 20% MCAR missing values
	Best: ('interp', 'time') (13.409 +- 0.719)
	Worst: ('mean', 'time') (91.556 +- 5.483)
Results with 50% MCAR missing values
	Best: ('interp', 'time') (16.178 +- 0.562)
	Worst: ('mean', 'time') (91.674 +- 3.060)
Results with 80% MCAR missing values
	Best: ('interp', 'time') (23.826 +- 2.998)
	Worst: ('mean', 'time') (92.711 +- 2.580)
Event 20120122-030734
Results with 10% MCAR missing values
	Best: ('interp', 'time') (293.382 +- 50.388)
	Worst: ('mean', 'time') (3469.700 +- 263.139)
Results with 20% MCAR missing values
	Best: ('interp', 'time') (320.465 +- 36.320)
	Worst: ('mean', 'time') (3468.067 +- 164.061)
Results with 50% MCAR missing values
	Best: ('interp', 'time') (471.700 +- 51.843)
	Worst: ('mean', 'time') (3492.265 +- 87.841)
Results with 80% MCAR missing values
	Best: ('interp', 'time') (832.514

In [92]:
df = pd.read_csv("/home/ojsant/gradu/results/20120102-153731.csv", header=[0, 1], index_col=[0, 1])
df

stat                      rmse_mean                                   \
miss_percent                     10         20         50         80   
method       axis                                                      
mean         time         92.757008  92.096808  91.729725  92.479686   
             pitch_angle  71.240574  71.894711  76.544442  86.776137   
fill_average time         13.176741  14.046641  17.492100  26.313820   
             pitch_angle  32.644831  36.645550  56.238944  80.077841   
interp       time         13.019381  13.632266  16.276658  24.316925   
             pitch_angle  30.661323  35.753789  55.013792  78.746280   
knn          time         31.417318  46.807198  63.312017  68.120953   
             pitch_angle  62.605294  66.280910  76.319294  85.551368   

stat                      rmse_std                                
miss_percent                    10        20        50        80  
method       axis                                                 
mean         time         8.348339  5.119441  2.720190  2.562565  
             pitch_angle  6.673050  5.212732  3.272944  4.947666  
fill_average time         1.005565  0.759810  0.870073  2.978261  
             pitch_angle  6.802828  5.529814  5.158456  4.927107  
interp       time         1.002679  0.670135  0.619182  2.981878  
             pitch_angle  5.185228  5.572225  5.744250  6.705853  
knn          time         6.298342  7.399690  3.563275  3.308399  
             pitch_angle  7.778915  5.362459  3.508322  4.244108

### KNNImputer (no blocks)

In [ ]:
from aipad.padimputer import PADImputer
from numpy.typing import NDArray
from os import PathLike

def KNN_traintest(file_path: PathLike, size_pct: float, seed: int) -> list:
    trues = []
    reduceds = []
    times = []
    mags = []
    intensities = []
    
    with h5py.File(file_path, 'r') as hdf:
        event_list = list(hdf.keys())
        
        for event_no in range(len(event_list)):
            event_data = load_event_data(hdf, event_list[event_no], wind)
            I_data = event_data["Intensity"]
            cov_data = event_data["Coverage"]
            B_data = event_data["MagField"]
            time = event_data["Intensity_attrs"]["Epoch"]
            X, Y, true = pad_histogram(wind, I_data.values, cov_data, 8)
            reduced_cov, mask = induce_missingness(cov_data, size_pct, seed)
            X, Y, reduced = pad_histogram(wind, I_data.values, reduced_cov, 8)

            trues.append(true)
            reduceds.append(reduced)
            times.append(time)
            mags.append(B_data)
            intensities.append(I_data)

    return make_train_test(trues, reduceds, times, mags, intensities, event_list)

neighbors = 5
n_splits = 3
miss_percent = 50
size_pct = miss_percent / 100

imputer = KNNImputer(n_neighbors=neighbors, keep_empty_features=True)

X_train, X_test, y_train, y_test, t_train, t_test, B_train, B_test, I_train, I_test, e_train, e_test = KNN_traintest(file_path, 0.5, 123)
X_train_stacked = np.vstack(X_train)

imputer = KNNImputer(n_neighbors=neighbors, weights="distance", keep_empty_features=True)
        
imputer.fit(X_train_stacked)
        
scores = []

for j in range(len(X_test)):
    path = Path(f"./rmse_random_sampling/{miss_percent}percent/{e_test[j]}.csv")
    
    true = X_test[j]
    reduced = y_test[j]
    imputed = imputer.transform(reduced)
    target, pred = target_from_prediction(true, reduced, imputed)
    score = calculate_scores(target, pred, "rmse")
    
    cols = ["method", "axis", "Mean", "Std"]

    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        df = pd.DataFrame(columns=cols)

    row = ["knn_with_traintest", 0, score, np.nan]

    df.loc[df.shape[0]] = row

    new_path = Path(f"./rmse_random_sampling/{miss_percent}percent/")
    new_path.mkdir(exist_ok=True)
    plot_dir = Path(f"./plots/imputers/knn_with_train_test/0")
    plot_dir.mkdir(exist_ok=True)
    pa = np.linspace(0.5*180/bins, (bins-0.5)*180/bins, bins)
    df.to_csv(new_path / f"{e_test[j]}.csv", index=False)
    plot_results(pa, I_test[j], B_test[j], true, reduced, imputed, "rmse", wind, solo,
                    save_path=plot_dir / f"{e_test[j]}_axis0_{miss_percent}percent")

IndexError: list index out of range

In [23]:
import os

imputer = PADImputer()
#os.chdir("./50percent")
print(f"{"Method".ljust(16)}{"Axis".ljust(5)}{"Mean".ljust(10)}\tStd")
for i, (method, axis) in enumerate(product(imputer._strategy_map.keys(), [0, 1])):
    means = []
    stds = []
    for file in os.listdir():
        if "csv" in file:
            df = pd.read_csv(file)
            means.append(df.iloc[i, 2])
            stds.append(df.iloc[i, 3])
        
    print(f"{method.ljust(16)}{str(axis).ljust(5)}{f"{np.nanmean(means):.3f}".rjust(10)}\t{np.nanmean(stds):.3f}")

Method          Axis Mean      	Std
mean            0      3361.873	394.617
mean            1      2398.780	307.752
fill_average    0       821.591	336.668
fill_average    1      2005.044	402.853
interp          0       742.480	340.712
interp          1      1920.949	395.762
knn             0      1595.137	304.984
knn             1      2404.533	321.397


### KNNImputer CV (with blocks)

In [ ]:
# Stacking of training cases for KNN imputer:
# concatenate 5 successive rows to one feature vector (5 * 180 = 900 features)
trains = []
for train in X_train:
    train_stacked = sliding_window_view(train, window_shape=(5,180)).reshape(-1, 5 * train.shape[1])   # ChatGPT solution
    trains.append(train_stacked)

X_train_stacked = np.vstack(trains)

In [ ]:
import logging
import sys
filehandler = logging.FileHandler(filename="./logs/cv_results.log", encoding="utf-8")
streamhandler = logging.StreamHandler(sys.stdout)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%m/%d/%Y %H:%M:%S', handlers=[filehandler, streamhandler], force=True)

def form_test_matrices_blocks(model: KNNImputer, X_test: np.ndarray,
                       y_test: np.ndarray, n_samples=10, sample_loc=120) -> list:
    true = X_test[sample_loc:sample_loc+n_samples]
    test = y_test[sample_loc:sample_loc+n_samples]
    reduced = np.where(np.isfinite(test), true, np.nan)
    
    reduced_reshaped = reduced.reshape((2, 900))
    pred_full_reshaped = model.transform(reduced_reshaped)
    pred_full = pred_full_reshaped.reshape((10, 180))

    pred = np.where((np.isnan(reduced)
                    & np.isfinite(true)
                    & np.meshgrid(np.isfinite(reduced).any(axis=1), np.arange(0, reduced.shape[1]),
                                  indexing="ij")[0]),
                    pred_full, np.nan)
    target = np.where(np.isfinite(pred), true, np.nan)

    return [true, reduced, pred_full, pred, target]

# Model selection CV
neighbors = np.arange(0, 31, 5)
neighbors[0] = 1
#weights = ["distance", "uniform"]      10.2. no noticeable difference between these
n_splits = 3
n_events = 20
n_samples = 10
sample_loc = 120 # onset is located at two hours after start of the window

# Cross validation: shift train-test split by one 3 times (a sort of 3-fold CV),
# but test only 10 samples in each event (after onset) and only 20 events in each fold

for n in neighbors:
    imputer = KNNImputer(n_neighbors=n, weights="distance", keep_empty_features=True)
    
    for shift in range(n_splits):
        logging.info(f"KNN neighbors={n}, split {shift}")
        result_df = pd.DataFrame()
        (X_train, X_test, y_train, y_test,
        I_train, I_test, meta_train, meta_test) = make_train_test(hist_arr, reduced_hist_arr,
                                                                intensity_arr, metadata_arr,
                                                                n=n_splits, shift=shift)
        trains = []
        for train in X_train:
            train_stacked = sliding_window_view(train, window_shape=(5, train.shape[1])).reshape(-1, 5 * train.shape[1])   # ChatGPT solution
            trains.append(train_stacked)

        X_train_stacked = np.vstack(trains)
        imputer.fit(X_train_stacked)
        scores = []
        for j in range(20):
            res = form_test_matrices_blocks(imputer, X_test[j], y_test[j])
            # plot_results(model, res, "rmse", I_test[j], sc=wind, cov_sc=solo,
            #              save_path=plot_path / "knnimputer" / "crossvalidation" / f"knn_{n}_{w}_split_{shift}_results_{j}.png")
            
            # Score on n_samples samples during main event
            score = calculate_scores(res[3],
                                     res[4], "rmse")
            result_df = save_results(imputer, res, meta_test[j], "rmse", Path(f"./knn_{n}_split_{shift}_results.csv"))
            scores.append(score)
            logging.info(f"Score for test event {j}: {score}")

        clear_output()
        logging.info(f"KNN neighbors={n}: CV score on fold {shift}={np.mean(scores):.4f}")


02/10/2026 12:46:53 KNN neighbors=15, weights=uniform: CV score on fold 2=2407.7693
